In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# classifier architecture
class EmbedWipeout(nn.Module):
    def __init__(self, h=28, w=28, d=28*28, l=10):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Flatten(),
            nn.Linear(h*w, d),
        )
        self.norm = nn.LayerNorm(d)
        self.wipeout = nn.Linear(d, l, bias=False)

    def forward(self, x):
        hidden = self.norm(self.embed(x))
        logits = self.wipeout(hidden)
        if self.training:
            return logits
        else:
            return F.softmax(logits, dim=1)

In [ ]:
# create classifier
model = EmbedWipeout(28,28,3,3).to(device)

In [ ]:
# get datasets
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
# select a subset of classes
classes_to_keep = [0, 6, 9]
class_to_id = {class_: i for i, class_ in enumerate(classes_to_keep)}
print(class_to_id)
id_to_class = {i: class_ for class_, i in class_to_id.items()}
print(id_to_class)

In [ ]:
class RemappedSubset(Dataset):
    def __init__(self, original_dataset, indices, class_to_new):
        self.original = original_dataset
        self.indices = indices
        self.class_to_new = class_to_new

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x, y = self.original[real_idx]
        new_y = self.class_to_new[int(y)]
        return x, new_y

In [ ]:
# create subset
train_indices = [i for i, (_, label) in enumerate(train_dataset) if label in classes_to_keep] # indices for train subset
test_indices = [i for i, (_, label) in enumerate(test_dataset) if label in classes_to_keep] # indices for test subset
train_subset = RemappedSubset(train_dataset, train_indices, class_to_id)
test_subset = RemappedSubset(test_dataset, test_indices, class_to_id)

In [ ]:
# test subset dataset
labels = torch.tensor([label for _, label in train_subset])
print(torch.unique(labels))
labels = torch.tensor([label for _, label in test_subset])
print(torch.unique(labels))

In [ ]:
# create train and test dataloader
batch_size = 512
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

In [ ]:
# Define loss funcion
criterion = nn.CrossEntropyLoss()

In [ ]:
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# scheduler
step_size = 8 #16
gamma = 0.5 #0.7
scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

In [ ]:
# training
history_loss = []
history_acc = []
history_test_acc = []
epoch = 0

In [ ]:
def train(num_epochs):
    global epoch
    for _ in range(num_epochs):
        # change model in training mood
        model.train()

        # to record loss and accuracy
        batch_loss = []
        total_train = 0
        correct_train = 0

        for batch, (x_train, y_train) in enumerate(train_loader):

            # send data to device
            input = x_train.to(device)

            # reset parameters gradient to zero
            optimizer.zero_grad()

            # forward pass to the model
            output = model(input)

            # categorization
            expected_output = y_train.to(device)

            # cross entropy loss
            loss = criterion(output, expected_output)

            # find gradients
            loss.backward()
            # update parameters using gradients
            optimizer.step()

            # recording loss
            batch_loss.append(loss.item())

            # recording accuracy
            total_train += output.shape[0]
            correct_train += torch.argmax(output,dim=1).to('cpu').eq(y_train).sum().item()

        epoch_loss = np.average(batch_loss)
        epoch_acc = (100.0 * correct_train) / total_train

        history_loss.append(epoch_loss)
        history_acc.append(epoch_acc)

        total_test = 0
        correct_test = 0

        model.eval()

        for batch, (x_test, y_test) in enumerate(test_loader):

            # send data to device
            input = x_test.to(device)

            # forward pass to the model
            with torch.no_grad():
                output = model(input)

            # recording accuracy
            total_test += output.shape[0]
            correct_test += torch.argmax(output,dim=1).to('cpu').eq(y_test).sum().item()

        test_acc = (100.0 * correct_test) / total_test

        history_test_acc.append(test_acc)

        print(f'Epoch: {epoch} Loss: {epoch_loss:.6f} Accuracy: {epoch_acc:.4f} Test accuracy: {test_acc:.4f} Learning Rate: {scheduler.get_last_lr()[0]:.7f}')
        scheduler.step()
        epoch += 1

In [ ]:
# training
print(f"number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
num_epochs = 16
train(num_epochs)

In [ ]:
model.eval()

In [ ]:
# Save model
def save(model_name):
    torch.save(model.state_dict(), model_name) # weights only
    print('Saved trained model at %s ' % model_name)

In [ ]:
save('mnist_classifier_embed-wipeout-069.pth')

In [ ]:
# download the model
from google.colab import files
files.download('mnist_classifier_embed-wipeout-069.pth')

In [ ]:
# test
sample = train_dataset[train_indices[0]]
print(sample[0].shape,sample[1])

In [ ]:
probabilities = model(sample[0].unsqueeze(0).to(device))
print(probabilities.argmax().item())

In [ ]:
sample_ = sample[0].squeeze(0).detach().numpy()
plt.imshow(sample_, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# representatives
vecs = model.wipeout.weight.detach().cpu().numpy()
print(vecs.shape)
print(vecs)

In [ ]:
traces = []
for i, vec in enumerate(vecs):
    x, y, z = vec.tolist()
    color = px.colors.qualitative.T10[i]
    label = id_to_class[i]
    traces.append(go.Scatter3d(x=[0, vec[0]], y=[0, vec[1]], z=[0, vec[2]], mode='lines', line=dict(color=color, width=5),name=str(id_to_class[i])))
    traces.append(go.Cone(x=[vec[0]], y=[vec[1]], z=[vec[2]], u=[vec[0]], v=[vec[1]], w=[vec[2]], colorscale=[[0, color], [1, color]], showscale=False, sizemode='absolute', sizeref=0.3))
fig = go.Figure(data=traces)
r = 1.5
fig.update_layout(scene=dict(xaxis=dict(range=[-r, r]), yaxis=dict(range=[-r, r]), zaxis=dict(range=[-r, r]), aspectmode='data'))
fig.show()


In [ ]:
sample = train_dataset[train_indices[1]]

In [ ]:
embedding = model.norm(model.embed(sample[0].to(device)))[0].detach().cpu().numpy()
print(embedding.shape)
print(embedding)

In [ ]:
embedding @ vecs.T

In [ ]:
id_to_class[np.argmax(embedding @ vecs.T)]

In [ ]:
sample_ = sample[0].squeeze(0).detach().numpy()
plt.close('all')
plt.imshow(sample_, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# collect embeddings for all samples from the testing dataset
embeddings = []
unnormalized_embeddings = []
categories = []
with torch.no_grad():
    for samples_x, samples_y in test_loader:
        unnormalized_embedding = model.embed(samples_x.to(device))
        unnormalized_embeddings.append(unnormalized_embedding)
        embedding = model.norm(unnormalized_embedding)
        embeddings.append(embedding)
        categories.append(samples_y)

In [ ]:
unnormalized_embeddings = torch.cat(unnormalized_embeddings, dim=0)
print(unnormalized_embeddings.shape)
embeddings = torch.cat(embeddings, dim=0)
print(embeddings.shape)
categories = torch.cat(categories, dim=0)
print(categories.shape)

In [ ]:
# calculate embeddings norms
norms = torch.norm(embeddings, dim=1)
norms = norms.detach().cpu().numpy()
print(norms.shape, norms.min(), norms.max(), norms.mean())

In [ ]:
# draw the norms distribution
plt.close('all')
plt.hist(norms, bins=20)
plt.xlabel('Norm')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# visualize latent space before normalization
points = unnormalized_embeddings.detach().cpu().numpy()
colors = px.colors.qualitative.T10
traces = []
for i in range(3):
    mask = categories == i
    pts = points[mask]
    scatter = go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers', marker=dict(size=5, color=colors[i]), name=id_to_class[i])
    traces.append(scatter)
fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z')),
    legend=dict(x=1.05, y=1),
    margin=dict(l=0, r=200, t=50, b=0)
)
fig.show()

In [ ]:
# visualize latent space after normalization
points = embeddings.detach().cpu().numpy()
colors = px.colors.qualitative.T10
traces = []
for i in range(3):
    mask = categories == i
    pts = points[mask]
    scatter = go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers', marker=dict(size=5, color=colors[i]), name=id_to_class[i])
    traces.append(scatter)
fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z')),
    legend=dict(x=1.05, y=1),
    margin=dict(l=0, r=200, t=50, b=0)
)
fig.show()